# 05 — Fairness: what removing sex and age did and did not achieve

**ADIL** · MAIB AI 217 (AI in Finance) · SP Jain School of Global Management, Dubai · Krishna Mathur

The project brief assumed real protected attributes were unavailable and that documented
proxies would stand in. That is wrong in both directions, and the correction runs through this
whole notebook.

`CODE_GENDER` is **sex**. `DAYS_BIRTH` is **age**. Both are genuinely protected under
essentially every consumer credit regime. This is a real fairness audit on real protected
attributes and it is reported as one — not as a methodological demonstration on proxies.

What is absent is **nationality and ethnicity**, which in a UAE context is the axis that
matters most. Home Credit offers no honest proxy for it. That dimension is declared
unaddressable here rather than approximated, and no result in this notebook speaks to it.

Neither attribute is a model input — deciding on sex or age is disparate treatment. This
notebook asks what that exclusion actually bought, and answers: less than it looks like.

Outputs `reports/fairness_report.md` and `metrics/fairness.json`.

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
from fairlearn.metrics import (
    MetricFrame,
    equalized_odds_difference,
    false_positive_rate,
    true_positive_rate,
)
from spine.fairness import calibration_by_group, disparity_ratio, group_metrics

from adil import paths
from adil import split as sp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)

processed = paths.processed_dir()
frame = pd.read_parquet(processed / "adil_frame.parquet")
splits = pd.read_parquet(processed / "split_index.parquet")["split"].values
r0_predictions = pd.read_parquet(processed / "r0_predictions.parquet")
challenger_predictions = pd.read_parquet(processed / "challenger_predictions.parquet")
is_test = splits == "test"

probability = {
    "R0": r0_predictions.loc[is_test, "prob_calibrated"].values,
    "R1": challenger_predictions.loc[is_test, "R1_prob_calibrated"].values,
    "R3": challenger_predictions.loc[is_test, "R3_prob_calibrated"].values,
    "R4": challenger_predictions.loc[is_test, "R4_prob_calibrated"].values,
}
default = frame.loc[is_test, "TARGET"].values
repaid = 1 - default
print(f"test applications: {len(default):,}   default rate {default.mean():.4f}")

## 1. The groups, and the framing

Sex has four `XNA` rows across the whole dataset. Four is not a group — every rate computed on
it would be a rounding artifact with a confidence interval wider than the scale. They stay in
the modelling data and are **excluded from the sex audit**, which is stated here rather than
done quietly.

The confusion-matrix framing matters and is easy to get backwards. The *action being audited
is approval*, so the selected class is "approved" and the outcome of interest is "would have
repaid". That makes:

- **selection rate** = approval rate — the four-fifths rule metric
- **TPR** = approval rate among applicants who would have repaid — equal opportunity
- **FPR** = approval rate among applicants who would have defaulted

Framing it the other way round, with default as the positive class, produces numbers that are
arithmetically fine and answer a question nobody asked.

In [ ]:
sex = frame.loc[is_test, "CODE_GENDER"].astype(str).values
age_years = (-frame.loc[is_test, "DAYS_BIRTH"] / 365.25).values
age_band = pd.cut(
    age_years,
    [0, 25, 35, 45, 55, 200],
    right=False,
    labels=["<25", "25-34", "35-44", "45-54", "55+"],
).astype(str)

usable_sex = sex != "XNA"
print(f"sex 'XNA' rows excluded from the sex audit: {int((~usable_sex).sum())}")

base_rates = pd.concat(
    [
        pd.DataFrame(
            {
                "attribute": "sex",
                "group": g,
                "n": int((sex[usable_sex] == g).sum()),
                "default rate": float(default[usable_sex][sex[usable_sex] == g].mean()),
            }
            for g in sorted(set(sex[usable_sex]))
        ),
        pd.DataFrame(
            {
                "attribute": "age band",
                "group": g,
                "n": int((age_band == g).sum()),
                "default rate": float(default[age_band == g].mean()),
            }
            for g in sorted(set(age_band))
        ),
    ]
).set_index(["attribute", "group"])
base_rates.round(5)

**The base rates differ, and that fact governs everything below.** Men default at roughly
half again the rate of women in this sample; applicants under 25 at more than twice the rate
of those over 55.

Unequal base rates are not a bug to be corrected — they are a property of the sample. But they
are the precondition of an impossibility result, and section 5 shows it operating rather than
citing it.

## 2. Disparity across the whole approval range

The cost-optimal cutoff is set in notebook 06. Borrowing it here would make the two notebooks
circular, and picking a different one arbitrarily would invite the obvious objection that the
fairness result was chosen along with the threshold.

So disparity is reported as a **curve over every approval rate**, and notebook 06 reads off
whatever point its cost matrix lands on. The curve is also the more honest artifact: a single
disparity number is a fact about a cutoff at least as much as about a model.

In [ ]:
APPROVAL_RATES = np.arange(0.50, 0.99, 0.02)


def approvals_at(scores, approval_rate):
    # Approve the safest `approval_rate` share: threshold on the risk score.
    cutoff = np.quantile(scores, approval_rate)
    return (scores <= cutoff).astype(int)


curve_rows = []
for rung, scores in probability.items():
    for approval_rate in APPROVAL_RATES:
        approved = approvals_at(scores, approval_rate)
        for attribute, labels, mask in [
            ("sex", sex, usable_sex),
            ("age band", age_band, np.ones(len(age_band), dtype=bool)),
        ]:
            rates = group_metrics(repaid[mask], approved[mask], labels[mask])
            curve_rows.append(
                {
                    "rung": rung,
                    "attribute": attribute,
                    "approval_rate": float(approval_rate),
                    "selection_disparity": disparity_ratio(rates, "selection_rate"),
                    "tpr_disparity": disparity_ratio(rates, "tpr"),
                    "fpr_disparity": disparity_ratio(rates, "fpr"),
                }
            )

disparity_curve = pd.DataFrame(curve_rows)
pivot = disparity_curve.pivot_table(
    index="approval_rate", columns=["attribute", "rung"], values="selection_disparity"
)
print("approval-rate disparity ratio (min group / max group), by approval rate")
pivot.round(3).iloc[::4]

## 3. Group rates at a reference cutoff

The curve is the finding; a table is easier to read. Fixed at a **90% approval rate** purely
as a reference point, chosen before looking at any disparity number and matching the
highest-risk-decile population used in notebook 04. Notebook 06 recomputes all of this at the
cost-optimal cutoff.

In [ ]:
REFERENCE_APPROVAL = 0.90

group_tables = {}
for rung, scores in probability.items():
    approved = approvals_at(scores, REFERENCE_APPROVAL)
    group_tables[(rung, "sex")] = group_metrics(
        repaid[usable_sex], approved[usable_sex], sex[usable_sex]
    )
    group_tables[(rung, "age band")] = group_metrics(repaid, approved, age_band)

print("R4, by sex — approval is the selected action, 'repaid' the positive outcome")
group_tables[("R4", "sex")].round(4)

In [ ]:
group_tables[("R4", "age band")].round(4)

In [ ]:
summary_rows = []
for (rung, attribute), table in group_tables.items():
    summary_rows.append(
        {
            "rung": rung,
            "attribute": attribute,
            "approval disparity": disparity_ratio(table, "selection_rate"),
            "TPR disparity": disparity_ratio(table, "tpr"),
            "FPR disparity": disparity_ratio(table, "fpr"),
            "min approval rate": float(table["selection_rate"].min()),
            "max approval rate": float(table["selection_rate"].max()),
        }
    )
disparity_summary = pd.DataFrame(summary_rows).set_index(["attribute", "rung"]).sort_index()
print(f"at a {REFERENCE_APPROVAL:.0%} overall approval rate")
print("the 0.8 four-fifths line is a US employment screening heuristic, not a legal")
print("standard anywhere this project applies, and it screens rather than decides.")
disparity_summary.round(4)

## 4. Calibration by group — the metric this project prioritises

Under unequal base rates you cannot have everything, so the choice of what to protect has to be
made explicitly and defended.

**This project prioritises calibration by group.** A group whose predicted probabilities run
consistently above its observed default rate is being charged for risk it does not carry, and
no threshold chosen downstream repairs that — the input to the decision is simply wrong for
those people. Every other fairness quantity is computed *from* the probability; if the
probability is wrong for a group, everything after it is too.

The equalised-odds gap is reported in full rather than optimised away.

In [ ]:
calibration_rows = []
for rung, scores in probability.items():
    for attribute, labels, mask in [
        ("sex", sex, usable_sex),
        ("age band", age_band, np.ones(len(age_band), dtype=bool)),
    ]:
        table = calibration_by_group(default[mask], scores[mask], labels[mask])
        for group, row in table.iterrows():
            calibration_rows.append(
                {
                    "rung": rung,
                    "attribute": attribute,
                    "group": group,
                    "n": int(row["n"]),
                    "mean predicted": row["mean_predicted"],
                    "observed": row["observed_rate"],
                    "gap": row["calibration_gap"],
                }
            )

calibration = pd.DataFrame(calibration_rows)
print("calibration gap = mean predicted minus observed; positive over-predicts risk")
calibration[calibration["rung"] == "R4"].set_index(["attribute", "group"])[
    ["n", "mean predicted", "observed", "gap"]
].round(5)

In [ ]:
worst_gap = (
    calibration.assign(absolute=lambda d: d["gap"].abs())
    .groupby(["rung", "attribute"])["absolute"]
    .max()
    .unstack()
)
print("largest absolute calibration gap in any group")
worst_gap.round(5)

## 5. The impossibility, demonstrated rather than cited

Calibration and equalised odds cannot both hold when base rates differ. That is a theorem, and
citing it is cheaper than showing it — so here it is operating on this model.

The demonstration has three steps. Confirm the model is close to calibrated within each group.
Observe that its error rates nonetheless differ across groups. Then force the error rates equal
and watch calibration break.

In [ ]:
scores = probability["R4"]
approved = approvals_at(scores, REFERENCE_APPROVAL)

sex_labels = sex[usable_sex]
sex_repaid, sex_default = repaid[usable_sex], default[usable_sex]
sex_scores, sex_approved = scores[usable_sex], approved[usable_sex]

before = MetricFrame(
    metrics={"TPR": true_positive_rate, "FPR": false_positive_rate},
    y_true=sex_repaid,
    y_pred=sex_approved,
    sensitive_features=sex_labels,
)
print("step 1 — calibration within group is close:")
print(
    calibration_by_group(sex_default, sex_scores, sex_labels)[
        ["mean_predicted", "observed_rate", "calibration_gap"]
    ]
    .round(5)
    .to_string()
)
print("")
print("step 2 — but the error rates are not equal:")
print(before.by_group.round(5).to_string())
print(
    f"  equalised-odds difference: "
    f"{equalized_odds_difference(sex_repaid, sex_approved, sensitive_features=sex_labels):.5f}"
)

In [ ]:
# Step 3: force equal approval rates among those who repaid (equal opportunity) by
# giving each group its own cutoff, then re-examine calibration.
target_tpr = float(before.by_group["TPR"].min())
forced = np.zeros_like(sex_approved)
group_cutoffs = {}
for group in np.unique(sex_labels):
    in_group = sex_labels == group
    repaid_scores = sex_scores[in_group & (sex_repaid == 1)]
    cutoff = float(np.quantile(repaid_scores, target_tpr))
    group_cutoffs[group] = cutoff
    forced[in_group] = (sex_scores[in_group] <= cutoff).astype(int)

after = MetricFrame(
    metrics={"TPR": true_positive_rate, "FPR": false_positive_rate},
    y_true=sex_repaid,
    y_pred=forced,
    sensitive_features=sex_labels,
)
forced_disparity = group_metrics(sex_repaid, forced, sex_labels)
eo_before_value = float(
    equalized_odds_difference(sex_repaid, sex_approved, sensitive_features=sex_labels)
)
eo_after_value = float(equalized_odds_difference(sex_repaid, forced, sensitive_features=sex_labels))

print(f"step 3 — equalising TPR at {target_tpr:.4f} requires a different cutoff per group:")
for group, cutoff in group_cutoffs.items():
    print(f"    {group}: approve when predicted default probability <= {cutoff:.5f}")
print("")
print(after.by_group.round(5).to_string())

gaps = pd.DataFrame(
    {
        "before": before.by_group.max() - before.by_group.min(),
        "after": after.by_group.max() - after.by_group.min(),
    }
)
print("")
print("gap between sexes, before and after forcing equal opportunity:")
print(gaps.round(5).to_string())
print("")
print(f"  equalised-odds difference : {eo_before_value:.5f} -> {eo_after_value:.5f}")
print(f"  approval-rate disparity   : {disparity_ratio(forced_disparity, 'selection_rate'):.4f}")
print(
    f"  cutoff spread between sexes: "
    f"{max(group_cutoffs.values()) - min(group_cutoffs.values()):.5f}"
)
print("")
print("The TPR gap is closed by construction. The FPR gap is not, and equalised odds")
print("requires both, so the residual difference above IS the FPR gap.")

Two things happened, and the second is the more interesting.

**The price is written on the face of it.** Two applicants with identical files and different
sexes now face different cutoffs. That is disparate treatment — not a subtle statistical
trade-off but deciding on a protected attribute, which in a credit context is the thing
actually prohibited.

**And it did not even work.** Equalising the true positive rate reduced the equalised-odds
difference but did not eliminate it, because equalised odds requires **both** TPR and FPR
parity and the residual above is the FPR gap. Buying equal opportunity did not buy equalised
odds. Closing the FPR gap as well would need a second cutoff per group pulling the opposite
way, and with different base rates no single per-group threshold satisfies both at once.

So the demonstration is stronger than the usual statement of it: the trade is not
"calibration or equalised odds, pick one". It is that under unequal base rates, thresholding a
calibrated score cannot deliver equalised odds at all — and the attempt costs disparate
treatment before it even falls short.

This is why notebook 06 constrains **one cutoff for everyone** and reports the disparity that
remains, rather than equalising the gap away. The gap is reported; it is not optimised.

## 6. So what did removing sex and age achieve?

The obvious challenge to this project's design: sex and age were kept out of the feature set,
so is the model fair now?

No. Exclusion prevents *disparate treatment* — the model cannot condition on sex directly. It
does nothing about *disparate impact*, because the information survives in correlated
features. The direct way to show that is to try to reconstruct the protected attribute from
exactly the features the model is allowed to use.

In [ ]:
from sklearn.metrics import roc_auc_score

from adil import challenger

r4_features = pd.read_parquet(processed / "r4_features.parquet")["feature"].tolist()
is_train = splits == "train"

proxy_results = []
for label_name, target_values in [
    ("sex is male", (frame["CODE_GENDER"].astype(str) == "M").astype(int).values),
    ("age under 35", ((-frame["DAYS_BIRTH"] / 365.25) < 35).astype(int).values),
]:
    keep_train = is_train & (frame["CODE_GENDER"].astype(str) != "XNA").values
    keep_test = is_test & (frame["CODE_GENDER"].astype(str) != "XNA").values
    probe = challenger.fit(
        frame.loc[keep_train], target_values[keep_train], r4_features, num_boost_round=150
    )
    predicted = probe.predict(challenger.design_matrix(frame.loc[keep_test], r4_features))
    proxy_results.append(
        {
            "reconstructing": label_name,
            "from": f"R4's {len(r4_features)} features",
            "test AUC": float(roc_auc_score(target_values[keep_test], predicted)),
            "base rate": float(target_values[keep_test].mean()),
        }
    )

proxies = pd.DataFrame(proxy_results).set_index("reconstructing")
proxies.round(4)

## 7. Persist

In [ ]:
payload = {
    "seed": sp.SEED,
    "protected_attributes_are_real": True,
    "note": (
        "CODE_GENDER is sex and DAYS_BIRTH is age; both are genuinely protected attributes, "
        "not proxies. Nationality and ethnicity are absent from Home Credit and are declared "
        "unaddressable rather than approximated."
    ),
    "excluded_from_sex_audit": {"XNA": int((~usable_sex).sum())},
    "prioritised_metric": "calibration by group",
    "reference_approval_rate": REFERENCE_APPROVAL,
    "base_rates": base_rates.reset_index().to_dict("records"),
    "disparity_at_reference": disparity_summary.reset_index().to_dict("records"),
    "calibration_by_group": calibration.to_dict("records"),
    "worst_absolute_calibration_gap": worst_gap.to_dict(),
    "impossibility_demonstration": {
        "attribute": "sex",
        "rung": "R4",
        "equalised_odds_before": eo_before_value,
        "equalised_odds_after_group_cutoffs": eo_after_value,
        "tpr_gap_before": float(before.by_group["TPR"].max() - before.by_group["TPR"].min()),
        "tpr_gap_after": float(after.by_group["TPR"].max() - after.by_group["TPR"].min()),
        "fpr_gap_before": float(before.by_group["FPR"].max() - before.by_group["FPR"].min()),
        "fpr_gap_after": float(after.by_group["FPR"].max() - after.by_group["FPR"].min()),
        "group_cutoffs": {str(k): float(v) for k, v in group_cutoffs.items()},
        "cutoff_spread": float(max(group_cutoffs.values()) - min(group_cutoffs.values())),
        "conclusion": (
            "Equalising the true positive rate required a different cutoff per sex, which is "
            "disparate treatment, and still did not deliver equalised odds: the TPR gap "
            "closes by construction, the FPR gap does not, and equalised odds requires both. "
            "Under unequal base rates, thresholding a calibrated score cannot deliver "
            "equalised odds at all. Notebook 06 therefore uses one cutoff for everyone and "
            "reports the residual disparity rather than optimising it away."
        ),
    },
    "proxy_audit": proxies.reset_index().to_dict("records"),
}
(paths.metrics_dir() / "fairness.json").write_text(
    json.dumps(payload, indent=2, default=float) + "\n"
)
disparity_curve.to_parquet(processed / "disparity_curve.parquet", index=False)
calibration.to_parquet(processed / "calibration_by_group.parquet", index=False)
print("wrote metrics/fairness.json")

In [ ]:
tpr_gap_before = float(before.by_group["TPR"].max() - before.by_group["TPR"].min())
tpr_gap_after = float(after.by_group["TPR"].max() - after.by_group["TPR"].min())
fpr_gap_before = float(before.by_group["FPR"].max() - before.by_group["FPR"].min())
fpr_gap_after = float(after.by_group["FPR"].max() - after.by_group["FPR"].min())
sex_calibration_gap = float(
    calibration[(calibration["rung"] == "R4") & (calibration["attribute"] == "sex")]["gap"]
    .abs()
    .max()
)
sex_proxy_auc = float(proxies.loc["sex is male", "test AUC"])
age_proxy_auc = float(proxies.loc["age under 35", "test AUC"])
eo_before = float(
    equalized_odds_difference(sex_repaid, sex_approved, sensitive_features=sex_labels)
)
eo_after = float(equalized_odds_difference(sex_repaid, forced, sensitive_features=sex_labels))
spread = max(group_cutoffs.values()) - min(group_cutoffs.values())

lines = [
    "# ADIL — fairness audit",
    "",
    "Generated by `notebooks/05_fairness.ipynb`. Every number is computed, not typed.",
    "",
    "MAIB AI 217 · SP Jain School of Global Management, Dubai · Krishna Mathur",
    "",
    "## These are real protected attributes",
    "",
    "The project brief assumed protected attributes were unavailable and that documented",
    "proxies would stand in. That is wrong in both directions.",
    "",
    "`CODE_GENDER` is **sex** and `DAYS_BIRTH` is **age**. Both are genuinely protected under",
    "essentially every consumer credit regime. This is a real fairness audit on real",
    "protected attributes and is reported as one.",
    "",
    "**Nationality and ethnicity are absent from Home Credit**, and in a UAE context that is",
    "the axis that matters most. No honest proxy for it exists in this data. That dimension",
    "is declared unaddressable rather than approximated, and nothing in this report speaks",
    "to it. Any UAE deployment would need that audit performed on data that carries the",
    "attribute.",
    "",
    f"Sex has {int((~usable_sex).sum())} `XNA` rows in the test split. That is not a group —",
    "every rate computed on it would be a rounding artifact. They remain in the modelling",
    "data and are excluded from the sex audit.",
    "",
    "## Framing",
    "",
    "The action audited is **approval**, so the selected class is 'approved' and the positive",
    "outcome is 'would have repaid'. Selection rate is the approval rate, TPR is the approval",
    "rate among applicants who would have repaid (equal opportunity), and FPR is the approval",
    "rate among applicants who would have defaulted.",
    "",
    "## Base rates",
    "",
    "| Attribute | Group | n | Default rate |",
    "|---|---|---:|---:|",
]
for (attribute, group), row in base_rates.iterrows():
    lines.append(f"| {attribute} | {group} | {int(row['n']):,} | {row['default rate']:.4f} |")
lines += [
    "",
    "The base rates differ, and that governs everything else. Unequal base rates are a",
    "property of the sample rather than a defect, but they are the precondition of the",
    "impossibility result below.",
    "",
    "## Disparity",
    "",
    "The cost-optimal cutoff is set in notebook 06; borrowing it here would make the two",
    "notebooks circular, and choosing another arbitrarily would invite the objection that",
    "the fairness result was picked along with the threshold. Disparity is therefore",
    "computed across the whole approval range (`disparity_curve.parquet`), and the table",
    f"below fixes a **{REFERENCE_APPROVAL:.0%} approval rate** as a reference point chosen",
    "before any disparity number was seen.",
    "",
    "The 0.8 four-fifths line is a US employment-screening heuristic, not a legal standard",
    "anywhere this project applies. It screens; it does not decide.",
    "",
    "| Attribute | Rung | Approval disparity | TPR disparity | FPR disparity |",
    "|---|---|---:|---:|---:|",
]
for (attribute, rung), row in disparity_summary.iterrows():
    lines.append(
        f"| {attribute} | {rung} | {row['approval disparity']:.4f} | "
        f"{row['TPR disparity']:.4f} | {row['FPR disparity']:.4f} |"
    )
lines += [
    "",
    "## Calibration by group — the prioritised metric",
    "",
    "Under unequal base rates not everything can hold at once, so the choice of what to",
    "protect is made explicitly. **This project prioritises calibration by group.** A group",
    "whose predicted probabilities run above its observed default rate is charged for risk",
    "it does not carry, and no downstream threshold repairs that — the input to the decision",
    "is wrong for those people. Every other fairness quantity is computed from the",
    "probability, so if the probability is wrong for a group, so is everything after it.",
    "",
    "Largest absolute calibration gap in any group:",
    "",
    "| Rung | " + " | ".join(worst_gap.columns) + " |",
    "|---|" + "---:|" * len(worst_gap.columns),
]
for rung, row in worst_gap.iterrows():
    lines.append(f"| {rung} | " + " | ".join(f"{v:.5f}" for v in row) + " |")
lines += [
    "",
    "## The impossibility, shown rather than cited",
    "",
    "Calibration and equalised odds cannot both hold when base rates differ. Demonstrated on",
    "R4 and sex, in three steps.",
    "",
    "1. The model is close to calibrated within each sex — the largest absolute gap is",
    f"   **{sex_calibration_gap:.5f}**.",
    f"2. Its error rates nonetheless differ: equalised-odds difference **{eo_before:.4f}**.",
    "3. Forcing equal opportunity requires a different cutoff for each sex:",
    "",
]
for group, cutoff in group_cutoffs.items():
    lines.append(f"    {group}: approve when predicted default probability <= {cutoff:.5f}")
lines += [
    "",
    "| Gap between sexes | Before | After |",
    "|---|---:|---:|",
    f"| TPR | {tpr_gap_before:.5f} | {tpr_gap_after:.5f} |",
    f"| FPR | {fpr_gap_before:.5f} | {fpr_gap_after:.5f} |",
    f"| Equalised-odds difference | {eo_before:.5f} | {eo_after:.5f} |",
    "",
    "Two things happened, and the second matters more.",
    "",
    f"**The price is explicit.** A cutoff spread of {spread:.5f} means two applicants with",
    "identical files and different sexes face different decisions. That is disparate",
    "treatment — not a subtle statistical trade-off but deciding on a protected attribute,",
    "which is the thing actually prohibited in a credit context.",
    "",
    "**And it did not even work.** The TPR gap closes by construction; the FPR gap does not,",
    "and equalised odds requires both — so the residual difference *is* the FPR gap. Buying",
    "equal opportunity did not buy equalised odds. Closing the FPR gap too would need a",
    "second per-group cutoff pulling the other way, and under unequal base rates no single",
    "threshold per group satisfies both.",
    "",
    "The demonstration is therefore stronger than the usual statement. The trade is not",
    "'calibration or equalised odds, pick one'. It is that under unequal base rates,",
    "thresholding a calibrated score cannot deliver equalised odds at all — and the attempt",
    "incurs disparate treatment before it even falls short.",
    "",
    "This is why notebook 06 uses **one cutoff for everyone** and reports the residual",
    "disparity. The equalised-odds gap is reported in full; it is not optimised away.",
    "",
    "## What excluding sex and age achieved",
    "",
    "Exclusion prevents **disparate treatment**: the model cannot condition on sex or age",
    "directly. It does nothing about **disparate impact**, because the information survives",
    "in correlated features. Shown directly, by trying to reconstruct each attribute from",
    "exactly the features R4 is permitted to use:",
    "",
    "| Reconstructing | From | Test AUC | Base rate |",
    "|---|---|---:|---:|",
]
for label, row in proxies.iterrows():
    lines.append(f"| {label} | {row['from']} | {row['test AUC']:.4f} | {row['base rate']:.4f} |")
lines += [
    "",
    f"Sex is recoverable at AUC **{sex_proxy_auc:.3f}** and age band at **{age_proxy_auc:.3f}**",
    "from the 20 features the model actually uses. The model does not need `CODE_GENDER`",
    "because it has effective substitutes for it.",
    "",
    "**So the honest answer is no — removing the attributes did not make the model fair.**",
    "It made the model lawful in one specific respect and left disparate impact untouched,",
    "which is exactly why the disparity above has to be measured and reported rather than",
    "assumed away by pointing at the excluded columns.",
    "",
    "## Limitations",
    "",
    "- Nationality and ethnicity are absent and are not proxied. For a UAE deployment this is",
    "  the material gap, and nothing here closes it.",
    "- Sex is recorded as a binary in this dataset. That is the data's limitation, reproduced",
    "  here because misrepresenting it would be worse, and it does not reflect the range of",
    "  people a real lending book serves.",
    "- Age bands are conventional cuts, not a modelling result. Different cuts would move the",
    "  disparity ratios.",
    "- Disparity ratios are point estimates on one test split with no interval. Groups with",
    "  few defaults carry the widest uncertainty and the narrowest reported number.",
    "- Public competition data, not UAE consumer data. The distribution differs materially",
    "  and no disparity figure here transfers to an Emirati lending book.",
    "",
]
path = paths.reports_dir() / "fairness_report.md"
path.write_text("\n".join(lines) + "\n")
print(f"wrote {path} ({len(lines)} lines)")